In [ ]:

import tsim
import stim
import pyzx as zx
import pyzx_param as param

In [ ]:
circ = tsim.Circuit(
"""
CX 0 1
CX 1 0
X 1
"""
)

In [ ]:
for x in circ:
    print(circ)
    g = circ.pop(index=0)
    x = g.targets_copy()
    circ.append_from_stim_program_text(g.name + " " + str(x[0].qubit_value) + " " + str(x[1].qubit_value))


In [ ]:
for t in range(len(circ)):
    print(1)

In [ ]:
len(circ)

In [ ]:
g = circ.pop(index=0)
print(g.tag)
print(g.name)
x = g.targets_copy()
x

In [ ]:
y = [0,0]

for i in range(0, len(x), 2):
                print(x[i].qubit_value)
                print(x[i + 1].qubit_value)

In [ ]:
z0 = 1
z1 = 0

#calculate probability
denom = abs(z0) ** 2 + abs(z1) ** 2
p = abs(z0) ** 2 / denom

p

In [ ]:
import numpy as np

In [ ]:
had_mat = np.array([[1, 1], [1, -1]], dtype=complex) / np.sqrt(2)

In [ ]:
def _bitstring_to_index(bits: list[int]) -> int:
    """Convert a bitstring [b_0, b_1, ..., b_{n-1}] to an integer index."""
    idx = 0
    for b in bits:
        idx = (idx << 1) | b
    return idx


In [ ]:
n_qubits = 2
state = np.zeros(2 ** n_qubits, dtype=complex)

state[0] = 1
print(state)

target = 0
state = state.reshape(2, n_qubits)
print(state)

# Move target axis to front for easy contraction
state = np.moveaxis(state, target, 0)
state = np.einsum("ij,j...->i...", had_mat, state)
state = np.moveaxis(state, 0, target)
print(state)
state = state.reshape(-1)
print(state)


In [ ]:
x_index = _bitstring_to_index([0,0])
complex(state[x_index])


In [ ]:
def gate_by_gate(circuit: tsim.Circuit):
    circ_final = "I"
    y = [0] * circuit.num_qubits
    for t in range(len(circuit)):
        gate = circ.pop(index=0)
        # Gate is a CNOT so update the output classically
        if gate.name == ("CX" or "CNOT" or "ZCX"):
            targets = gate.targets_copy()
            for i in range(0, len(targets), 2):
                a = targets[i].qubit_value
                b = targets[i + 1].qubit_value
                if y[a] == 1:
                    y[b] = 1 - y[b]
        if gate.name == "H":
    return y

In [ ]:
gate_by_gate(circ)


In [ ]:

import tsim
import stim
import pyzx as zx
import gate_by_gate as gbg
import tsim
import numpy as np
from fractions import Fraction
import pyzx_param as param

In [ ]:

x = [0, 0]

test_circ = tsim.Circuit("""

    H 0
    CX 0 1
    """)
num_qubits = test_circ.num_qubits
# test_circ.append_from_stim_program_text(f"R {' '.join(str(q) for q in range(num_qubits))}")
g = test_circ.diagram("pyzx")

# For each qubit, find the vertex with the highest row number
last_vertices = {}
for v in g.vertices():
    q = g.qubit(v)
    if q not in last_vertices or g.row(v) > g.row(last_vertices[q]):
        last_vertices[q] = v

print(last_vertices)
zx.draw(g, labels=True)

In [ ]:
for qubit, bit in enumerate(x):
    out_vertex = last_vertices[qubit]
    phase = Fraction(0) if bit == 0 else Fraction(1)  # 0 = |0>, pi = |1>
    # Insert a Z-spider with the right phase before the output
    g.set_type(out_vertex, zx.VertexType.Z)
    g.set_phase(out_vertex, phase)

In [ ]:

zx.draw(g, labels=True)

In [ ]:
param.full_reduce(g, paramSafe=True)

In [ ]:
complex(g.scalar.to_number())

In [ ]:
num_qubits = 4
circ_until_now = tsim.Circuit()

circ_until_now.append_from_stim_program_text(f"R {' '.join(str(q) for q in range(num_qubits))}")
circ_until_now.append_from_stim_program_text(f"R {' '.join(str(q) for q in range(num_qubits))}")

circ_until_now

In [ ]:
g = circ_until_now.diagram("pyzx")


In [ ]:
zx.full_reduce(g)
zx.draw(g)

In [ ]:
g.outputs()

In [ ]:
import tsim
import stim
import pyzx as zx
import gate_by_gate as gbg
import tsim
import numpy as np
from fractions import Fraction
import pyzx_param as param

In [ ]:
print("=== Bell-state circuit ===")
# Produces |Φ+> = (|00> + |11>) / sqrt(2)
# Expected: samples should be 00 or 11 with equal probability

bell_circuit = tsim.Circuit(
"""
H 0
CX 0 1
"""
)

# (|000> + |111>) / sqrt(2) — should only see 000 or 111
ghz_circuit = tsim.Circuit(
    """
    H 0
    CX 0 1 0 2
    """
)

x_circuit = tsim.Circuit(
    """
    X 0
    CX 0 1
    """
)

example_circuit = tsim.Circuit(
    """
    H 0
    CX 0 1
    H 1
    CX 0 1
    H 0
    """
)

reset_circuit = tsim.Circuit(
    """
    X 0 1 2 3
    CX 0 1
    R 0
    """
)

resetX_circuit = tsim.Circuit(
    """
    RX 0
    """
)

check_circuit = tsim.Circuit(
    """
    X 0
    H 0
    H 0
    """
)

measure_circ = tsim.Circuit("""
    RX 0
    MX 0 1
""")

rng = np.random.default_rng(42)
counts = {"00": 0, "11": 0, "01": 0, "10": 0}
N = 1000
detects = None
for _ in range(N):
    passed, result, detects = gbg.gate_by_gate(measure_circ.copy(), detects)
    if passed:
        key = "".join(map(str, result))
        counts[key] = counts.get(key, 0) + 1

print(f"Samples from {N} runs:")
for k, v in sorted(counts.items()):
    if v > 0:
        print(f"  |{k}>: {v} ({100*v/N:.1f}%)")


In [ ]:
bell_circuit = tsim.Circuit(
"""
R 0
H 0
"""
)

In [132]:
uggy = param.Graph()

In [133]:
uggy.add_vertex(ty = zx.VertexType.X)

0

In [121]:
uggy.add_vertex(ty = zx.VertexType.X)
uggy.add_edge([0,1], zx.EdgeType.HADAMARD)

[0, 1]

In [134]:
uggy.add_params(0, {'a'})

In [114]:
uggy.add_params(1, {'b'})

In [135]:
param.draw(uggy, labels=True)

In [136]:
param.full_reduce(uggy)
param.draw(uggy, labels=True)

In [138]:
uggy.scalar.phasenodevars

[{'a'}]

In [ ]:
g = bell_circuit.diagram("pyzx")


In [ ]:
zx.draw(g, labels=True)

In [ ]:
g.scalar

In [221]:
bell_circuit = tsim.Circuit(
"""
R 0
"""
)

In [222]:
g = bell_circuit.diagram("pyzx")

In [223]:
last_vertices = {}
for v in g.vertices():
    q = g.qubit(v)
    if q not in last_vertices or g.row(v) > g.row(last_vertices[q]):
        last_vertices[q] = v

In [224]:
print(last_vertices)
x = [0]

{0: 1}


In [225]:
for qubit, bit in enumerate(x):
    out_vertex = last_vertices[qubit]
    # phase = Fraction(0) if bit == 0 else Fraction(1)  # 0 = |0>, pi = |1>
    # Insert a Z-spider with the right phase before the output
    g.set_type(out_vertex, zx.VertexType.X)
    if qubit == 0: g.add_params(out_vertex, 'a')
    else: g.add_params(out_vertex, 'b')

In [226]:
param.draw(g, labels=True)

In [229]:
for x in g.vertices():
    print(x, g.get_params(x))


0 set()
1 {'a'}


In [230]:
param.full_reduce(g, paramSafe=True)
param.draw(g, labels=True)

In [231]:
g.scalar.phasevars_halfpi

{}

In [232]:
g.scalar.phasevars_pi_pair

[]

In [233]:
g.scalar.phasevars_pi

set()

In [234]:
g.scalar.phasenodevars

[{'a'}]

In [237]:
'a' in g.scalar.phasenodevars[0]

True

In [ ]:
00, 01, 10 , 11

In [ ]:
zx.draw(g, labels=True)

In [ ]:
param.full_reduce(g, paramSafe=True)

In [ ]:
amplitude = complex(g.scalar.to_number())
a = amplitude

In [ ]:
a

In [ ]:
amplitude = complex(g.scalar.to_number())
a = amplitude

In [ ]:
x = [0]

In [ ]:
for qubit, bit in enumerate(x):
    out_vertex = last_vertices[qubit]
    phase = Fraction(0) if bit == 0 else Fraction(1)  # 0 = |0>, pi = |1>
    # Insert a Z-spider with the right phase before the output
    g.set_type(out_vertex, zx.VertexType.X)
    g.set_phase(out_vertex, phase)

In [ ]:
bell_circuit = tsim.Circuit(
"""
X_ERROR(0.1) 0
"""
)

In [ ]:
g = bell_circuit.pop(index=0)
g.gate_args_copy()

In [ ]:
zx.draw(g, labels=True)

In [ ]:
for gate in bell_circuit:
    targets = gate.targets_copy()
    print(targets)

In [ ]:
bell_circuit = tsim.Circuit(
"""
R 0
H 0
"""
)